# 03 · Fase 2 — Modelo final, evaluación y visualización

Entrena el modelo ganador sobre los 452k estudiantes completos y genera
todos los artefactos para el artículo.

> Equivalente interactivo de `scripts/run_fase2_modelo_final.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')
import joblib, numpy as np

from saberpro_clustering.config import load_config
from saberpro_clustering.preprocessing import load_raw_data, preprocesar_saber_pro
from saberpro_clustering.dimensionality import reducir_umap
from saberpro_clustering.clustering import entrenar_modelo_final
from saberpro_clustering.evaluation import calcular_metricas, ablation_study, validar_representatividad
from saberpro_clustering.visualization import plot_clusters_2d, calcular_heatmap_caracterizacion, plot_heatmap_caracterizacion
from saberpro_clustering.geo import analizar_geografia, plot_geo_heatmap, top_departamentos_por_cluster

config = load_config('../config.yaml')
f2 = config['fase2']

## Preprocesar los 452k completos y calcular UMAP

In [ ]:
df = load_raw_data(config['data']['input_path'])
df_limpio_full, df_filtrado_full, X_full, _, encoder, scaler = preprocesar_saber_pro(df, sample_n=None)
X_umap_full, reducer = reducir_umap(X_full, n_components=f2['best_ndim'])

## Entrenar modelo final

In [ ]:
modelo, labels_final = entrenar_modelo_final(
    X_umap_full, best_algo=f2['best_algo'], best_k=f2['best_k'],
    dbscan_eps=f2['dbscan_eps'], dbscan_min_samples=f2['dbscan_min_samples'],
    hdbscan_mc=f2['hdbscan_min_cluster_size'], hdbscan_ms=f2['hdbscan_min_samples']
)
df_filtrado_full['_cluster'] = labels_final
m_final = calcular_metricas(X_umap_full, labels_final)
m_final

## Ablation study (justificación de UMAP)

In [ ]:
df_ablation = ablation_study(X_full, labels_final, m_final, f2['best_ndim'])
df_ablation

## Visualización — embedding UMAP 2D

In [ ]:
nombres_clusters = {i: f'Cluster {i}' for i in range(f2['best_k'])}
fig = plot_clusters_2d(X_umap_full, labels_final, nombres=nombres_clusters,
                        guardar='../reports/figures/embedding_final.png')

## Heatmap de caracterización por clúster

Muestra las asimetrías de educación de los padres y demás variables entre clústeres.

In [ ]:
df_heat = calcular_heatmap_caracterizacion(df_filtrado_full, cluster_column='_cluster')
clusters_ids = sorted(c for c in df_filtrado_full['_cluster'].unique() if c != -1)
plot_heatmap_caracterizacion(df_heat, clusters_ids, nombres_clusters,
                              guardar='../reports/figures/heatmap_clusters.png')

## Análisis geográfico

In [ ]:
tabla_geo = analizar_geografia(df_filtrado_full, cluster_column='_cluster',
                                 col_geo=config['data']['col_geo'], nombres_clusters=nombres_clusters)
if tabla_geo is not None:
    plot_geo_heatmap(tabla_geo, guardar='../reports/figures/geo_clusters.png')
    top_departamentos_por_cluster(tabla_geo)

## Validación de representatividad (200k vs 452k)

In [ ]:
_, df_filtrado_200k, *_ = preprocesar_saber_pro(df, sample_n=config['fase1']['sample_n'])
df_rep = validar_representatividad(df_limpio_full, df_filtrado_200k)
df_rep